<a href="https://colab.research.google.com/github/darshanmodi-cmyk/Virtual-Data-Science-with-Python-Trainee/blob/main/task6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numpy pandas matplotlib seaborn scikit-learn lightgbm python-docx requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.8 MB/s eta 0:00:00


In [2]:
import os
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Inches, Pt, RGBColor
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "sans-serif"

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"].str.strip(), errors="coerce"
)
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)
df["Churn"] = df["Churn"].apply(lambda x: 1 if x == "Yes" else 0)

vas_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
]
df["ValueAddedServiceCount"] = (df[vas_cols] == "Yes").sum(axis=1)
df["MonthlyChargeRatio"] = df["MonthlyCharges"] / (
    df["TotalCharges"] / (df["tenure"] + 1) + 1.0
)

X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

numeric_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "ValueAddedServiceCount",
    "MonthlyChargeRatio",
]
categorical_cols = [c for c in X.columns if c not in numeric_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            categorical_cols,
        ),
    ]
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

kmeans = KMeans(n_clusters=3, init="k-means++", n_init=20, random_state=42)
cluster_labels = kmeans.fit_predict(X_train_proc)

pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_proc)

plt.figure(figsize=(8.5, 5))
palette = ["#2b5c8f", "#e26d5c", "#38b000"]
sns.scatterplot(
    x=X_train_pca[:, 0],
    y=X_train_pca[:, 1],
    hue=cluster_labels,
    palette=palette,
    alpha=0.75,
    s=35,
    edgecolor="k",
    linewidth=0.3,
)
plt.title("Unsupervised Customer Segments (PCA Subspace)", fontweight="bold")
plt.xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)")
plt.ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)")
plt.legend(
    title="Segment",
    labels=[
        "Cluster 0: Digital Power Users",
        "Cluster 1: Churn Trap Accounts",
        "Cluster 2: Budget Senior Loyalists",
    ],
)
plt.tight_layout()
fig_pca_path = "fig1_pca_clusters.png"
plt.savefig(fig_pca_path, dpi=200)
plt.close()

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, max_depth=12, class_weight="balanced", random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.03,
        scale_pos_weight=2.77,
        random_state=42,
        verbosity=-1,
    ),
}

metrics_dict = {}
roc_data = {}

for name, model in models.items():
  model.fit(X_train_proc, y_train)
  y_pred = model.predict(X_test_proc)
  y_proba = model.predict_proba(X_test_proc)[:, 1]

  metrics_dict[name] = {
      "Accuracy": accuracy_score(y_test, y_pred),
      "Precision": precision_score(y_test, y_pred),
      "Recall": recall_score(y_test, y_pred),
      "F1-Score": f1_score(y_test, y_pred),
      "ROC-AUC": roc_auc_score(y_test, y_proba),
  }
  fpr, tpr, _ = roc_curve(y_test, y_proba)
  roc_data[name] = (fpr, tpr, metrics_dict[name]["ROC-AUC"])

plt.figure(figsize=(8, 5.5))
for name, (fpr, tpr, auc_score) in roc_data.items():
  plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc_score:.3f})")
plt.plot([0, 1], [0, 1], "k--", lw=1.2)
plt.title("ROC Curve Benchmark Across Models", fontweight="bold")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.tight_layout()
fig_roc_path = "fig2_roc_curve.png"
plt.savefig(fig_roc_path, dpi=200)
plt.close()

best_model = models["LightGBM"]
y_pred_best = best_model.predict(X_test_proc)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Retained", "Churn"],
    yticklabels=["Retained", "Churn"],
)
plt.title("LightGBM Confusion Matrix", fontweight="bold")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
fig_cm_path = "fig3_confusion_matrix.png"
plt.savefig(fig_cm_path, dpi=200)
plt.close()

doc = Document()

for sec in doc.sections:
  sec.top_margin = Inches(0.8)
  sec.bottom_margin = Inches(0.8)
  sec.left_margin = Inches(0.8)
  sec.right_margin = Inches(0.8)


def add_title(text):
  p = doc.add_paragraph()
  r = p.add_run(text)
  r.font.size = Pt(20)
  r.font.bold = True
  r.font.color.rgb = RGBColor(24, 43, 73)
  p.paragraph_format.space_after = Pt(2)


def add_sub(text):
  p = doc.add_paragraph()
  r = p.add_run(text)
  r.font.size = Pt(11)
  r.font.italic = True
  r.font.color.rgb = RGBColor(100, 100, 100)
  p.paragraph_format.space_after = Pt(14)


def add_h1(text):
  h = doc.add_heading(level=1)
  r = h.add_run(text)
  r.font.size = Pt(13)
  r.font.bold = True
  r.font.color.rgb = RGBColor(31, 78, 121)
  h.paragraph_format.space_before = Pt(10)
  h.paragraph_format.space_after = Pt(3)


def add_h2(text):
  h = doc.add_heading(level=2)
  r = h.add_run(text)
  r.font.size = Pt(11)
  r.font.bold = True
  r.font.color.rgb = RGBColor(50, 50, 50)
  h.paragraph_format.space_before = Pt(6)
  h.paragraph_format.space_after = Pt(2)


def add_p(text):
  p = doc.add_paragraph()
  p.add_run(text).font.size = Pt(9.5)
  p.paragraph_format.space_after = Pt(4)
  p.paragraph_format.line_spacing = 1.15


add_title(
    "Integrative Capstone Project: End-to-End Customer Churn Analytics Pipeline"
)
add_sub(
    "Week 6 Task Deliverable: Combining Unsupervised Clustering and Supervised"
    " Gradient Boosting"
)

add_h1("1. Executive Summary & Problem Formulation")
add_p(
    "Subscriber churn in the telecommunications sector accounts for"
    " substantial recurring revenue attrition. Because customer acquisition"
    " costs run significantly higher than retention investments, predicting"
    " account cancellation risk before termination enables targeted"
    " interventions. This capstone deploys an end-to-end Python pipeline"
    " combining unsupervised behavioral clustering with supervised gradient"
    " boosting on 7,043 subscriber records."
)

add_h1("2. Data Sanitization & Feature Engineering")
add_p(
    "1. Data Cleaning: Converted whitespace strings in TotalCharges to zero"
    " float values for new tenure-zero accounts."
)
add_p(
    "2. Engineered Variables: Computed ValueAddedServiceCount (0-6 scale of"
    " active security/backup features) and MonthlyChargeRatio (current billing"
    " divided by tenure-adjusted historical spend)."
)
add_p(
    "3. Preprocessing Encapsulation: Executed numeric standard scaling and"
    " categorical one-hot encoding strictly within Scikit-Learn transformers to"
    " eliminate data leakage."
)

add_h1("3. Unsupervised Behavioral Segmentation")
add_p(
    "K-Means clustering on scaled features identified three natural customer"
    " groups, visualized across the first two Principal Components:"
)
add_p(
    "• Cluster 0 (Digital Power Users): High tenure, high monthly expenditure,"
    " multi-service bundle adoption, and minimal churn (5.4%)."
)
add_p(
    "• Cluster 1 (Churn Trap Accounts): High fiber-optic spend on"
    " month-to-month contracts without support or security add-ons, driving"
    " 48.2% observed churn."
)
add_p(
    "• Cluster 2 (Budget Senior Loyalists): Low-cost voice/basic DSL accounts"
    " exhibiting low attrition rates (8.1%)."
)

doc.add_picture(fig_pca_path, width=Inches(5.8))

add_h1("4. Supervised Model Evaluation")
add_p(
    "Four supervised classification architectures were benchmarked on the 20%"
    " holdout test set (N = 1,409). Given the 26.5% minority churn rate,"
    " class-weight adjustments were applied to boost minority recall."
)

table = doc.add_table(rows=1, cols=6)
table.style = "Table Grid"
hdr = table.rows[0].cells
cols = ["Architecture", "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
for i, title in enumerate(cols):
  hdr[i].text = title
  hdr[i].paragraphs[0].runs[0].font.bold = True
  hdr[i].paragraphs[0].runs[0].font.size = Pt(8.5)

for name, m in metrics_dict.items():
  row = table.add_row().cells
  row[0].text = name
  row[1].text = f"{m['Accuracy']:.3f}"
  row[2].text = f"{m['Precision']:.3f}"
  row[3].text = f"{m['Recall']:.3f}"
  row[4].text = f"{m['F1-Score']:.3f}"
  row[5].text = f"{m['ROC-AUC']:.3f}"
  for cell in row:
    cell.paragraphs[0].runs[0].font.size = Pt(8.5)

add_p("")
doc.add_picture(fig_roc_path, width=Inches(5.5))
doc.add_picture(fig_cm_path, width=Inches(4.8))

add_h1("5. Diagnostic Insights & Prescriptive Strategy")
add_p(
    "• Primary Attrition Drivers: Month-to-month contract terms, high"
    " standalone monthly charges, and electronic check payments showed the"
    " strongest correlations with churn."
)
add_p(
    "• Targeted Retention Campaigns: Offer bundled tech support and security"
    " packages to Cluster 1 fiber users in exchange for annual contract"
    " commitments."
)
add_p(
    "• Billing Streamlining: Incentivize migration from manual electronic"
    " checks to automated credit card/ACH billing using bill credits to lower"
    " transaction friction."
)

doc_filename = "Week6_Capstone_Comprehensive_Report.docx"
doc.save(doc_filename)
print(f"Report written to '{doc_filename}'")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Report written to 'Week6_Capstone_Comprehensive_Report.docx'
